# ThreadLine — Demand Forecasting & Capacity Planning for Garment Manufacturing

An end-to-end decision-support system: forecasts team-level demand (WIP for sewing,
overtime for finishing), then solves a mixed-integer program to recommend daily worker
allocation and overtime, validated against real historical outcomes.

**Headline result:** on a realistic walk-forward backtest, this system could have saved
**~72%** of scheduling cost versus what actually happened historically (hindsight ceiling: ~77%).
XGBoost forecasting beats a naive baseline (MAPE 0.31 vs. 0.57) and an LSTM benchmark (0.78).

**Dashboard:** see `app.py` — run with `streamlit run app.py` (not part of this notebook's
sequential cells; Streamlit apps need a running server context, not inline execution).

**Phases in this notebook:** 1-3 (audit/clean/EDA) → 4 (forecasting) → 5 (MILP optimization)
→ 6 (pipeline integration) → 7 (scenario analysis) → 9 (impact validation).
Phase 8 (dashboard) is `app.py`. Phase 10 (this repo) is portfolio packaging.

## Setup

Run this cell first. Installs the two dependencies not preinstalled in most
Colab/local environments, and defines the shared paths every phase below uses.

In [1]:
!pip install -q pulp xgboost

import os

# Path to the raw dataset, relative to this notebook's working directory.
DATA_PATH = "../data/garments_worker_productivity.csv"
OUTPUT_DIR = "../outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

## Phase 1-3 — Data Audit, Cleaning & Feature Engineering, EDA

Audits the raw CSV for the issues that actually matter (the `team`/`team_id` collision, finishing's structural lack of a WIP signal), builds the department-aware `demand_proxy`, and runs the volatility/correlation EDA.

In [2]:
"""
ThreadLine — Phases 1-3: Data Audit, Cleaning & Feature Engineering, EDA & Segmentation
==========================================================================================
Run directly:
    python phase1_3_audit_eda.py path/to/garments_worker_productivity.csv [output_dir]

CORRECTIONS vs. the original draft (validated against the actual 1,197-row dataset):

1. IDEMPOTENCY BUG: the original merged freshly-built wip_roll3/wip_lag1 columns
   back onto the dataframe without checking whether those names already existed.
   Run it on already-cleaned data (or run it twice) and pandas silently creates
   wip_roll3_x / wip_roll3_y / wip_lag1_x / wip_lag1_y instead of erroring.
   Confirmed: running the original on this dataset bloated it from 26 to 28 columns.
   Fix: ENGINEERED_COLS are dropped before recompute, every time.

2. DEMAND-SIGNAL BUG: `workload_pressure = wip.fillna(0)` looks like a reasonable
   missing-value fill, but it's not filling in missing data -- WIP is 100% absent
   for the entire finishing department. Filling with 0 doesn't approximate a real
   value, it manufactures a constant, which silently zeroes out every finishing
   team in the volatility ranking (Q1), the day-of-week effect, and the driver
   correlation matrix.
   Checked which raw fields actually vary for finishing: wip, no_of_style_change,
   idle_time, and idle_men are ALL exactly constant-zero there. over_time is not
   (std ~1777) -- it's the only workload-adjacent field with real variance for
   that department, so it's used as finishing's demand proxy.
   Fix: `demand_proxy` = wip for sewing rows, over_time for finishing rows. Every
   downstream EDA/segmentation function operates on demand_proxy so the fix
   propagates automatically instead of needing to be special-cased everywhere.
"""

import sys
import pandas as pd
import numpy as np

# Columns this script generates. Dropped before recompute so reruns on
# already-processed files are idempotent instead of producing _x/_y duplicates.
ENGINEERED_COLS = [
    "team_num", "team_id", "over_target_flag", "workload_pressure",
    "demand_proxy", "demand_proxy_source", "style_change_flag",
    "rated_capacity_min", "idle_loss_min", "effective_capacity_min",
    "productivity_gap", "wip_roll3", "wip_lag1",
    "demand_proxy_roll3", "demand_proxy_lag1",
]


# ==========================================================================
# PHASE 1 — DATA AUDIT
# ==========================================================================
def audit_dataset(raw_csv_path):
    """Run the Phase 1 data quality audit and return (df, findings)."""
    df = pd.read_csv(raw_csv_path)
    findings = {}

    findings["shape"] = df.shape
    findings["duplicates"] = int(df.duplicated().sum())

    # --- date integrity ---
    date_parsed = pd.to_datetime(df["date"], errors="coerce")
    findings["unparseable_dates"] = int(date_parsed.isnull().sum())
    findings["date_range"] = (str(date_parsed.min().date()), str(date_parsed.max().date()))
    findings["unique_dates"] = int(date_parsed.nunique())

    # --- completeness ---
    missing = df.isnull().sum()
    findings["missing_counts"] = missing[missing > 0].to_dict()

    # --- categorical consistency ---
    findings["department_raw_values"] = df["department"].unique().tolist()
    findings["quarter_values"] = sorted(df["quarter"].unique().tolist())
    findings["day_values"] = sorted(df["day"].unique().tolist())

    dept_clean = df["department"].str.strip().str.replace("sweing", "sewing", regex=False)

    # --- team identity check: does 'team' repeat across departments? ---
    sewing_teams = set(df.loc[dept_clean == "sewing", "team"])
    finishing_teams = set(df.loc[dept_clean == "finishing", "team"])
    findings["team_numbers_shared_across_departments"] = sorted(sewing_teams & finishing_teams)

    # --- actual_productivity range check ---
    findings["productivity_over_1_count"] = int((df["actual_productivity"] > 1.0).sum())
    findings["productivity_max"] = float(df["actual_productivity"].max())

    # --- over_time scale check: team-total or per-worker? ---
    findings["overtime_vs_teamsize_corr"] = round(float(df["over_time"].corr(df["no_of_workers"])), 3)
    safe_workers = df["no_of_workers"].replace(0, np.nan)
    findings["overtime_per_worker_median"] = round(float((df["over_time"] / safe_workers).median()), 1)

    # --- GENERIC per-department zero-variance scan ---
    # Rather than hardcoding "wip is the problem field", scan every numeric
    # column and flag any that are exactly constant (std == 0, or fully null)
    # within a department. This is what actually surfaces the demand-signal bug.
    numeric_cols = [c for c in [
        "targeted_productivity", "smv", "wip", "over_time", "incentive",
        "idle_time", "idle_men", "no_of_style_change", "no_of_workers",
        "actual_productivity",
    ] if c in df.columns]

    dead_by_dept = {}
    std_by_dept = df.groupby(dept_clean)[numeric_cols].std()
    for dept in std_by_dept.index:
        dead_cols = [c for c in numeric_cols
                     if pd.isna(std_by_dept.loc[dept, c]) or std_by_dept.loc[dept, c] == 0]
        dead_by_dept[dept] = dead_cols
    findings["zero_variance_fields_by_department"] = dead_by_dept
    findings["std_by_department"] = std_by_dept.round(3)

    # --- recommend a demand proxy per department: highest-std non-dead field
    # among the fields that plausibly represent workload pressure ---
    workload_candidates = [c for c in ["wip", "over_time", "no_of_style_change"] if c in df.columns]
    proxy_recommendation = {}
    for dept in std_by_dept.index:
        usable = {c: std_by_dept.loc[dept, c] for c in workload_candidates
                  if not pd.isna(std_by_dept.loc[dept, c]) and std_by_dept.loc[dept, c] > 0}
        proxy_recommendation[dept] = max(usable, key=usable.get) if usable else None
    findings["recommended_demand_proxy_by_department"] = proxy_recommendation

    return df, findings


def print_audit_report(findings):
    print("=" * 72)
    print("PHASE 1 — DATA AUDIT REPORT")
    print("=" * 72)
    print(f"Shape: {findings['shape'][0]} rows x {findings['shape'][1]} columns")
    print(f"Duplicate rows: {findings['duplicates']}")
    print(f"Unparseable dates: {findings['unparseable_dates']}")
    print(f"Date range: {findings['date_range'][0]} to {findings['date_range'][1]} "
          f"({findings['unique_dates']} unique dates)")
    print()
    print("Missing values (columns with >0 nulls):")
    for col, n in findings["missing_counts"].items():
        print(f"  - {col}: {n}")
    if not findings["missing_counts"]:
        print("  none")
    print()
    print(f"Raw department values: {findings['department_raw_values']}")
    print("  -> trim + typo fix ('sweing' -> 'sewing') applied before use")
    print()
    print(f"Team numbers shared across departments: {findings['team_numbers_shared_across_departments']}")
    print("  -> 'team' alone is NOT a unique identifier. Use department+team as the key.")
    print()
    print(f"actual_productivity > 1.0: {findings['productivity_over_1_count']} rows "
          f"(max {findings['productivity_max']:.3f}) -- kept, not clipped; flagged instead.")
    print()
    print(f"over_time correlation with no_of_workers: {findings['overtime_vs_teamsize_corr']} "
          f"(median per-worker overtime: {findings['overtime_per_worker_median']} min)")
    print("  -> over_time is a TEAM-TOTAL figure. Any per-team overtime bound must scale with team size.")
    print()
    print("Zero-variance fields by department (std == 0 or fully null):")
    for dept, cols in findings["zero_variance_fields_by_department"].items():
        print(f"  - {dept}: {cols if cols else 'none'}")
    print("  -> a field that's dead-constant within a department carries NO signal there.")
    print("     Using it anyway (e.g. filling missing wip with 0) manufactures a fake")
    print("     constant instead of real data -- it will silently zero out that")
    print("     department in any downstream volatility/correlation analysis.")
    print()
    print(f"Recommended demand proxy per department: {findings['recommended_demand_proxy_by_department']}")
    print("  -> chosen as the highest-variance non-dead field among {wip, over_time, no_of_style_change}.")
    print("=" * 72)


# ==========================================================================
# PHASE 2 — DATA ENGINEERING (cleaning + feature construction)
# ==========================================================================
def clean_pipeline(raw_csv_path):
    """Clean the raw CSV and engineer demand/capacity/cost features. Idempotent."""
    df = pd.read_csv(raw_csv_path)
    df.columns = df.columns.str.strip()

    # idempotency: drop any pre-existing engineered columns before recomputing,
    # so this is safe to run on raw data OR on a previous run's output.
    df = df.drop(columns=[c for c in ENGINEERED_COLS if c in df.columns])

    # categorical cleanup
    df["department"] = df["department"].str.strip().str.replace("sweing", "sewing", regex=False)
    df["date"] = pd.to_datetime(df["date"])

    # composite team identity ('team' repeats across departments)
    df["team_num"] = df["team"]
    df["team_id"] = df["department"] + "_" + df["team"].astype(str)

    # actual_productivity: NOT clipped -- rows above 1.0 look like genuine
    # over-target performance, flagged instead of discarded
    df["over_target_flag"] = (df["actual_productivity"] > 1.0).astype(int)

    df["style_change_flag"] = (df["no_of_style_change"] > 0).astype(int)

    # capacity / cost features
    df["rated_capacity_min"] = df["no_of_workers"] * df["smv"]
    df["idle_loss_min"] = df["idle_time"] * df["idle_men"]
    df["effective_capacity_min"] = df["rated_capacity_min"] - df["idle_loss_min"]
    df["productivity_gap"] = df["targeted_productivity"] - df["actual_productivity"]

    # workload_pressure: WIP with missing filled to 0. Kept only as a raw
    # numeric convenience field -- NOT used for segmentation/correlation,
    # because it's constant-zero for the entire finishing department (WIP
    # isn't tracked there) and mixing that into a cross-department metric
    # silently erases finishing's signal. See demand_proxy below instead.
    df["workload_pressure"] = df["wip"].fillna(0)

    # demand_proxy: the department-appropriate workload signal.
    # sewing has real WIP variance -> use wip.
    # finishing has zero variance on wip, no_of_style_change, idle_time and
    # idle_men (confirmed on this dataset) -- over_time is the only workload-
    # adjacent field that actually varies there, so it stands in as the proxy.
    df["demand_proxy"] = np.where(df["department"] == "sewing", df["wip"], df["over_time"])
    df["demand_proxy_source"] = np.where(df["department"] == "sewing", "wip", "over_time")

    # calendar-aware lag/rolling features per team_id, computed on demand_proxy
    # so both departments get a meaningful rolling/lag feature for Phase 4,
    # instead of finishing getting rolling stats over a column that's always 0.
    df = df.merge(
        _time_aware_rolling_features(df, value_col="demand_proxy", prefix="demand_proxy"),
        on=["team_id", "date"], how="left",
    )
    # keep wip-specific lag/roll too, since sewing's forecasting target is wip itself
    df = df.merge(
        _time_aware_rolling_features(df, value_col="workload_pressure", prefix="wip"),
        on=["team_id", "date"], how="left",
    )

    return df.sort_values(["team_id", "date"]).reset_index(drop=True)


def _time_aware_rolling_features(df, value_col, prefix):
    """3-day rolling mean and 1-day lag of value_col, per team_id, aligned to the
    full observed date grid (so a gap day isn't mislabeled as 'yesterday')."""
    all_dates = pd.Index(sorted(df["date"].unique()), name="date")
    frames = []
    for tid, g in df.groupby("team_id"):
        s = g.set_index("date")[value_col].reindex(all_dates)
        frames.append(pd.DataFrame({
            "team_id": tid,
            "date": all_dates,
            f"{prefix}_roll3": s.rolling(3, min_periods=1).mean().values,
            f"{prefix}_lag1": s.shift(1).values,
        }))
    return pd.concat(frames, ignore_index=True)


# ==========================================================================
# PHASE 3 — EDA & SEGMENTATION
# ==========================================================================
def segment_volatility(df):
    """Coefficient-of-variation segmentation using demand_proxy, which is
    already department-appropriate (wip for sewing, over_time for finishing) --
    so this naturally produces a meaningful ranking for BOTH departments,
    instead of finishing collapsing to NaN/0."""
    vol = df.groupby(["department", "team_id"])["demand_proxy"].agg(["mean", "std"])
    vol["cv"] = vol["std"] / vol["mean"].replace(0, np.nan)
    return vol.sort_values("cv", ascending=False)


def day_of_week_effect(df):
    """Split by department -- combining wip-scale sewing values with
    over_time-scale finishing values in one global average would still
    conflate two different units."""
    return df.pivot_table(index="day", columns="department", values="demand_proxy", aggfunc="mean")


def quarter_effect(df):
    return df.groupby("quarter")["actual_productivity"].mean()


def driver_correlations(df, department=None):
    """Correlation matrix of workload/capacity/cost drivers against
    productivity. Computed per department by default (sewing and finishing
    are different processes with likely different drivers); pass
    department=None for the combined view."""
    cols = ["demand_proxy", "no_of_workers", "smv", "over_time", "incentive",
            "idle_time", "style_change_flag", "targeted_productivity", "actual_productivity"]
    data = df if department is None else df[df["department"] == department]
    return data[cols].corr()


def style_change_impact(df):
    """Restricted to sewing: style_change_flag is definitionally always 0
    for finishing, so including those rows would just dilute the '0' bucket
    with rows that were never at risk of a style change in the first place."""
    sewing = df[df["department"] == "sewing"]
    return sewing.groupby("style_change_flag")["actual_productivity"].mean()


def overtime_productivity_relationship(df):
    return round(float(df["over_time"].corr(df["actual_productivity"])), 3)


def run_eda(df, output_dir=None):
    results = {
        "volatility_segmentation": segment_volatility(df),
        "day_of_week_effect": day_of_week_effect(df),
        "quarter_effect": quarter_effect(df),
        "correlation_matrix_sewing": driver_correlations(df, "sewing"),
        "correlation_matrix_finishing": driver_correlations(df, "finishing"),
        "style_change_impact": style_change_impact(df),
        "overtime_productivity_corr": overtime_productivity_relationship(df),
    }

    if output_dir:
        results["correlation_matrix_sewing"].to_csv(f"{output_dir}/correlation_matrix_sewing.csv")
        results["correlation_matrix_finishing"].to_csv(f"{output_dir}/correlation_matrix_finishing.csv")
        results["volatility_segmentation"].to_csv(f"{output_dir}/volatility_segmentation.csv")
        try:
            import matplotlib
            matplotlib.use("Agg")
            import matplotlib.pyplot as plt
            import seaborn as sns

            fig, axes = plt.subplots(1, 2, figsize=(15, 6))
            for ax, dept in zip(axes, ["sewing", "finishing"]):
                sns.heatmap(results[f"correlation_matrix_{dept}"], annot=True, fmt=".2f",
                            cmap="coolwarm", center=0, ax=ax)
                ax.set_title(f"Driver Correlation Matrix — {dept}")
            plt.tight_layout()
            plt.savefig(f"{output_dir}/correlation_heatmap.png", dpi=150)
            plt.close(fig)
        except ImportError:
            print("(matplotlib/seaborn not available -- skipped heatmap image)")

    return results


def print_eda_report(results):
    print("=" * 72)
    print("PHASE 3 — EDA & SEGMENTATION REPORT")
    print("=" * 72)
    print("\nTeam volatility ranking (CV of demand_proxy; top 5 per department):")
    for dept, g in results["volatility_segmentation"].groupby(level=0):
        print(f"\n  {dept}:")
        print(g.droplevel(0).head(5).round(3).to_string())
    print("\nAverage demand_proxy by day of week, split by department:")
    print(results["day_of_week_effect"].round(2))
    print("\nAverage actual productivity by quarter:")
    print(results["quarter_effect"].round(3))
    print(f"\nOvertime vs. productivity correlation (all rows): {results['overtime_productivity_corr']}")
    print("\nSewing correlation matrix:")
    print(results["correlation_matrix_sewing"].round(2))
    print("\nFinishing correlation matrix:")
    print(results["correlation_matrix_finishing"].round(2))
    print("\nProductivity by style-change flag, sewing only (0 = no change, 1 = style changed):")
    print(results["style_change_impact"].round(3))
    print("=" * 72)


# ==========================================================================
# MAIN
# ==========================================================================

raw_path = DATA_PATH
output_dir = OUTPUT_DIR

# Phase 1
_, audit_findings = audit_dataset(raw_path)
print_audit_report(audit_findings)

# Phase 2
clean_df = clean_pipeline(raw_path)
print(f"\nPhase 2 complete — cleaned dataset shape: {clean_df.shape}")
if output_dir:
    clean_df.to_csv(f"{output_dir}/garments_worker_productivity_CLEANED.csv", index=False)
    print(f"Saved cleaned dataset to {output_dir}/garments_worker_productivity_CLEANED.csv")
print()

# Phase 3
eda_results = run_eda(clean_df, output_dir=output_dir)
print_eda_report(eda_results)

# alias for downstream phases
df = clean_df.copy()


PHASE 1 — DATA AUDIT REPORT
Shape: 1197 rows x 26 columns
Duplicate rows: 0
Unparseable dates: 0
Date range: 2015-01-01 to 2015-03-11 (59 unique dates)

Missing values (columns with >0 nulls):
  - wip: 506
  - wip_lag1: 106

Raw department values: ['finishing', 'sewing']
  -> trim + typo fix ('sweing' -> 'sewing') applied before use

Team numbers shared across departments: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
  -> 'team' alone is NOT a unique identifier. Use department+team as the key.

actual_productivity > 1.0: 37 rows (max 1.120) -- kept, not clipped; flagged instead.

over_time correlation with no_of_workers: 0.734 (median per-worker overtime: 120.0 min)
  -> over_time is a TEAM-TOTAL figure. Any per-team overtime bound must scale with team size.

Zero-variance fields by department (std == 0 or fully null):
  - finishing: ['wip', 'idle_time', 'idle_men', 'no_of_style_change']
  - sewing: none
  -> a field that's dead-constant within a department carries NO signal there.
     Usi

PHASE 3 — EDA & SEGMENTATION REPORT

Team volatility ranking (CV of demand_proxy; top 5 per department):

  finishing:
                 mean       std     cv
team_id                               
finishing_1  1888.163  2055.058  1.088
finishing_2  1983.462  2151.369  1.085
finishing_3  2774.595  2813.411  1.014
finishing_5  2115.429  2055.942  0.972
finishing_4  2758.696  2408.402  0.873

  sewing:
               mean       std     cv
team_id                             
sewing_4   1218.678  2912.877  2.390
sewing_2   1326.281  2737.033  2.064
sewing_3   1409.293  2667.081  1.892
sewing_10  1528.333  2717.816  1.778
sewing_1   1609.196  2105.399  1.308

Average demand_proxy by day of week, split by department:
department  finishing   sewing
day                           
Monday        1606.27  2156.31
Saturday      1892.53  1003.02
Sunday        1770.34  1016.20
Thursday      2866.67   974.03
Tuesday       1779.76  1012.48
Wednesday     1637.53   973.77

Average actual productivity by

## Phase 4 — Demand Forecasting

Naive baseline vs. XGBoost vs. LSTM, with a guarded MAPE (unguarded MAPE on this data returns literal `inf` for rows with zero true demand), P50/P90 quantile forecasts with monotonicity enforced, and an asymmetric business-cost comparison.

In [3]:
"""
ThreadLine — Phase 4: Demand Forecasting
==========================================================================================
Run directly:
    python phase4_forecasting.py path/to/garments_worker_productivity_CLEANED.csv [output_dir]

Expects the CLEANED output of phase1_3_audit_eda.py (needs team_id, demand_proxy,
demand_proxy_roll3, demand_proxy_lag1 to already exist).

CORRECTIONS vs. the draft (each one validated against the actual dataset before fixing):

1. TARGET VARIABLE BUG: the draft forecasts `workload_pressure`, which is exactly
   the department-blind column flagged as unsafe in Phase 1-3 -- it's constant 0
   for the entire finishing department. Forecasting it directly doesn't just
   under-inform the finishing side, it actively breaks the headline metric:
   sklearn's MAPE divides by |y_true|, so on rows where y_true == 0 it explodes
   toward its epsilon-guarded ceiling instead of contributing 0 error. Tested this
   on the actual data: MAPE computed on workload_pressure directly returns
   187,236,267,972,568,640 -- not a typo, an actual float that size, entirely an
   artifact of 10-ish structural zeros in the target. Switched to `demand_proxy`
   (department-appropriate, from Phase 1-3) and guarded MAPE to exclude the small
   number of remaining true-zero rows, documented below. Guarded MAPE on the real
   data comes out to a sane 0.57 for the naive baseline.

2. GROUPBY BUG: `test.groupby('team')` groups by the raw team NUMBER, which
   repeats across departments (team 1 exists as both sewing_1 and finishing_1).
   Shifting within that group blends two unrelated teams' histories together as
   if they were one continuous series. Fixed to group by `team_id`.

3. NO EXPLICIT TRAIN/TEST SPLIT: the draft references `train`/`test` without
   showing how they're built. Built here as a chronological split (last 12 of
   the dataset's 59 dates held out, ~21% of rows, every team represented) --
   never a random split, since this is a time series.

4. LAG-FEATURE FILL BUG: `fillna(0)` on `wip_lag1`-type features invents a fake
   "zero backlog" value for each team's first observed day, which is a real gap
   in the series, not a true zero. Rows missing the lag feature are dropped from
   both train and test instead (only ~24 rows, one per team).

5. CRASH: `test['naive_pred'].fillna(method='bfill')` raises
   `TypeError: NDFrame.fillna() got an unexpected keyword argument 'method'` on
   the pandas version in this environment (`method=` was removed) -- confirmed
   by running it. Fixed to `.bfill()`. More importantly, back-filling a pooled
   column across team boundaries would borrow one team's value for another
   team's missing first day -- fixed by dropping unmatched rows instead, and by
   evaluating the naive baseline and XGBoost on the exact same row set so the
   comparison in Q3 is apples-to-apples.

6. LSTM BUG: `df[df['team']==1]` has the same team/team_id collision as #2 --
   it silently concatenates sewing_1 and finishing_1 into one fake series before
   building sequences. It was also trained but never evaluated on held-out data,
   so there was no actual number to compare against XGBoost for Q4. Fixed to
   build windows per team_id, respect the same chronological cutoff as the other
   models, scale using train-only statistics (LSTM is scale-sensitive; the raw
   values here run into the thousands), and report the same MAPE/RMSE on the
   same held-out period.

7. QUANTILE MONOTONICITY: P50 and P90 are trained as two independent models, so
   nothing stops P90 from landing below P50 on a given row -- confirmed this
   happens on 36 of 235 test rows. Enforced `p90 = max(p50, p90)` afterward,
   since the optimizer downstream needs a valid buffer, not an occasional
   negative one.

8. Added per-department MAPE/RMSE breakdown throughout, since sewing (wip-scale)
   and finishing (over_time-scale) are different processes -- a single pooled
   number can hide one department being modeled far worse than the other.
"""

import sys
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder, StandardScaler
import xgboost as xgb

WINDOW = 3          # LSTM sequence length -- short on purpose, given ~59 dates of history
N_TEST_DATES = 12   # chronological holdout size (~20% of the 59 available dates)

FEATURES = [
    "no_of_workers", "smv", "over_time", "incentive", "idle_time",
    "style_change_flag", "demand_proxy_roll3", "demand_proxy_lag1",
    "targeted_productivity",
]


# ==========================================================================
# GUARDED METRICS
# ==========================================================================
def guarded_mape(y_true, y_pred):
    """MAPE, excluding rows where the true value is exactly 0 -- those rows
    make sklearn's MAPE divide by its epsilon floor and explode, dominating
    the metric with noise instead of signal. Reports how many rows were
    excluded so it's never a silent adjustment."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    mask = y_true > 0
    excluded = int((~mask).sum())
    if mask.sum() == 0:
        return np.nan, excluded
    return float(mean_absolute_percentage_error(y_true[mask], y_pred[mask])), excluded


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def business_cost(y_true, y_pred, under_penalty=3, over_penalty=1):
    """Asymmetric cost: under-forecasting (missed deadline / backlog) is
    penalized harder than over-forecasting (idle capacity), per Section 6.1."""
    error = np.asarray(y_true) - np.asarray(y_pred)
    cost = np.where(error > 0, error * under_penalty, -error * over_penalty)
    return float(cost.sum())


# ==========================================================================
# SPLIT
# ==========================================================================
def time_split(df, n_test_dates=N_TEST_DATES):
    dates = sorted(df["date"].unique())
    cutoff = dates[-n_test_dates]
    train = df[df["date"] < cutoff].copy()
    test = df[df["date"] >= cutoff].copy()
    return train, test, cutoff


# ==========================================================================
# NAIVE BASELINE
# ==========================================================================
def naive_baseline(df, train, test):
    """Yesterday's demand_proxy = today's forecast, computed within team_id so
    two different teams' histories never bleed into each other."""
    full_sorted = df.sort_values(["team_id", "date"])
    lagged = full_sorted.assign(
        naive_pred=full_sorted.groupby("team_id")["demand_proxy"].shift(1)
    )[["team_id", "date", "naive_pred"]]

    test_eval = test.merge(lagged, on=["team_id", "date"], how="left").dropna(subset=["naive_pred"])
    return test_eval


# ==========================================================================
# XGBOOST
# ==========================================================================
def fit_xgboost(train, test):
    le_dept = LabelEncoder()
    le_team = LabelEncoder()
    # fit encoders on the full label space so train/test never see an unseen code
    le_dept.fit(pd.concat([train["department"], test["department"]]))
    le_team.fit(pd.concat([train["team_id"], test["team_id"]]))

    def add_codes(d):
        d = d.copy()
        d["dept_enc"] = le_dept.transform(d["department"])
        d["team_enc"] = le_team.transform(d["team_id"])
        return d

    train = add_codes(train)
    test = add_codes(test)

    feature_cols = FEATURES + ["dept_enc", "team_enc"]

    # drop rows missing the lag feature (each team's first observed day)
    # instead of fillna(0), which would fabricate a fake "no backlog" history
    train_clean = train.dropna(subset=["demand_proxy_lag1"])
    test_clean = test.dropna(subset=["demand_proxy_lag1"])

    X_train, y_train = train_clean[feature_cols], train_clean["demand_proxy"]
    X_test, y_test = test_clean[feature_cols], test_clean["demand_proxy"]

    model = xgb.XGBRegressor(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, random_state=42,
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    test_clean = test_clean.assign(xgb_pred=pred)
    return model, test_clean, feature_cols


# ==========================================================================
# LSTM (secondary benchmark)
# ==========================================================================
def build_sequences(df, cutoff, window=WINDOW):
    """Per-team_id sliding windows of demand_proxy. A window/target pair is
    assigned to test if its TARGET date falls on/after cutoff -- inputs always
    come from dates strictly before the target, so there's no leakage even for
    the first test-period predictions."""
    train_X, train_y, test_X, test_y = [], [], [], []
    for _, g in df.sort_values("date").groupby("team_id"):
        vals = g["demand_proxy"].values
        dts = g["date"].values
        for i in range(len(vals) - window):
            w, target, target_date = vals[i:i + window], vals[i + window], dts[i + window]
            if pd.Timestamp(target_date) >= pd.Timestamp(cutoff):
                test_X.append(w); test_y.append(target)
            else:
                train_X.append(w); train_y.append(target)
    return (np.array(train_X), np.array(train_y),
            np.array(test_X), np.array(test_y))


def fit_lstm(df, cutoff):
    from tensorflow import keras
    from tensorflow.keras import layers
    import tensorflow as tf
    tf.random.set_seed(42)

    X_train, y_train, X_test, y_test = build_sequences(df, cutoff)

    # scale using TRAIN-ONLY statistics -- LSTM with tanh activation is
    # scale-sensitive, and these values run into the thousands unscaled
    scaler = StandardScaler().fit(y_train.reshape(-1, 1))
    X_train_s = scaler.transform(X_train.reshape(-1, 1)).reshape(X_train.shape)
    X_test_s = scaler.transform(X_test.reshape(-1, 1)).reshape(X_test.shape)
    y_train_s = scaler.transform(y_train.reshape(-1, 1)).ravel()

    model = keras.Sequential([
        keras.Input(shape=(WINDOW, 1)),
        layers.LSTM(16, activation="tanh"),
        layers.Dense(1),
    ])
    model.compile(optimizer="adam", loss="mse")
    model.fit(X_train_s.reshape(-1, WINDOW, 1), y_train_s, epochs=50, verbose=0)

    pred_s = model.predict(X_test_s.reshape(-1, WINDOW, 1), verbose=0).ravel()
    pred = scaler.inverse_transform(pred_s.reshape(-1, 1)).ravel()
    return y_test, pred


# ==========================================================================
# QUANTILE FORECASTS (P50 / P90)
# ==========================================================================
def fit_quantiles(train_clean, test_clean, feature_cols):
    X_train, y_train = train_clean[feature_cols], train_clean["demand_proxy"]
    X_test = test_clean[feature_cols]

    model_p50 = xgb.XGBRegressor(objective="reg:quantileerror", quantile_alpha=0.5, n_estimators=200)
    model_p90 = xgb.XGBRegressor(objective="reg:quantileerror", quantile_alpha=0.9, n_estimators=200)
    model_p50.fit(X_train, y_train)
    model_p90.fit(X_train, y_train)

    p50 = model_p50.predict(X_test)
    p90_raw = model_p90.predict(X_test)
    violations = int((p90_raw < p50).sum())
    p90 = np.maximum(p50, p90_raw)  # enforce P90 >= P50; independently-trained
                                     # quantile models don't guarantee this on their own

    return test_clean.assign(forecast_p50=p50, forecast_p90=p90), violations


# ==========================================================================
# MAIN
# ==========================================================================

output_dir = OUTPUT_DIR
# df is reused directly from the Phase 1-3 cell above -- already has
# team_id, demand_proxy, demand_proxy_roll3/lag1 built in
train, test, cutoff = time_split(df)
print(f"Train/test split at {pd.Timestamp(cutoff).date()}: "
      f"{len(train)} train rows, {len(test)} test rows, "
      f"{test['team_id'].nunique()} teams represented in test.")

# --- naive baseline ---
naive_eval = naive_baseline(df, train, test)
naive_mape, excl = guarded_mape(naive_eval["demand_proxy"], naive_eval["naive_pred"])
print(f"\nNaive baseline MAPE: {naive_mape:.3f} ({excl} zero-true-value rows excluded)")
print(f"Naive baseline RMSE: {rmse(naive_eval['demand_proxy'], naive_eval['naive_pred']):.2f}")

# --- xgboost ---
model, test_clean, feature_cols = fit_xgboost(train, test)
xgb_mape, xgb_excl = guarded_mape(test_clean["demand_proxy"], test_clean["xgb_pred"])
print(f"\nXGBoost MAPE: {xgb_mape:.3f} ({xgb_excl} zero-true-value rows excluded)")
print(f"XGBoost RMSE: {rmse(test_clean['demand_proxy'], test_clean['xgb_pred']):.2f}")

print("\nPer-department breakdown (XGBoost):")
for dept, g in test_clean.groupby("department"):
    d_mape, d_excl = guarded_mape(g["demand_proxy"], g["xgb_pred"])
    print(f"  {dept}: MAPE={d_mape:.3f} RMSE={rmse(g['demand_proxy'], g['xgb_pred']):.2f} "
          f"n={len(g)} ({d_excl} excluded)")

if output_dir:
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(8, 6))
        xgb.plot_importance(model, importance_type="gain", ax=ax)
        plt.tight_layout()
        plt.savefig(f"{output_dir}/xgb_feature_importance.png", dpi=150)
        plt.close(fig)
        print(f"\nSaved feature importance plot to {output_dir}/xgb_feature_importance.png")
    except ImportError:
        print("(matplotlib not available -- skipped feature importance plot)")

# --- lstm secondary benchmark ---
print("\nTraining LSTM (secondary benchmark, expected to underperform XGBoost "
      "given ~59 days of history)...")
y_test_seq, lstm_pred = fit_lstm(df, cutoff)
lstm_mape, lstm_excl = guarded_mape(y_test_seq, lstm_pred)
print(f"LSTM MAPE: {lstm_mape:.3f} ({lstm_excl} zero-true-value rows excluded)")
print(f"LSTM RMSE: {rmse(y_test_seq, lstm_pred):.2f}")

# --- quantile forecasts ---
# rebuild encoders the same way fit_xgboost did, so codes stay consistent
le_dept = LabelEncoder().fit(pd.concat([train["department"], test["department"]]))
le_team = LabelEncoder().fit(pd.concat([train["team_id"], test["team_id"]]))
train_q = train.dropna(subset=["demand_proxy_lag1"]).copy()
train_q["dept_enc"] = le_dept.transform(train_q["department"])
train_q["team_enc"] = le_team.transform(train_q["team_id"])

test_q, violations = fit_quantiles(train_q, test_clean, feature_cols)
print(f"\nQuantile forecasts built. P90<P50 violations before enforcement: "
      f"{violations}/{len(test_q)} (corrected with max(P50,P90)).")

# --- asymmetric business cost, naive vs xgboost, same aligned row set ---
aligned = test_clean.merge(
    naive_eval[["team_id", "date", "naive_pred"]], on=["team_id", "date"], how="inner"
)
naive_cost = business_cost(aligned["demand_proxy"], aligned["naive_pred"])
xgb_cost = business_cost(aligned["demand_proxy"], aligned["xgb_pred"])
print(f"\nBusiness cost (Naive):   {naive_cost:,.0f}")
print(f"Business cost (XGBoost): {xgb_cost:,.0f}")
print(f"Cost reduction vs. naive: {(1 - xgb_cost / naive_cost) * 100:.1f}%")

if output_dir:
    test_q.to_csv(f"{output_dir}/phase4_forecasts.csv", index=False)
    print(f"\nSaved forecast table to {output_dir}/phase4_forecasts.csv")


Train/test split at 2015-02-26: 943 train rows, 254 test rows, 24 teams represented in test.

Naive baseline MAPE: 0.574 (10 zero-true-value rows excluded)
Naive baseline RMSE: 1206.23

XGBoost MAPE: 0.306 (10 zero-true-value rows excluded)
XGBoost RMSE: 462.51

Per-department breakdown (XGBoost):
  finishing: MAPE=0.165 RMSE=686.40 n=99 (10 excluded)
  sewing: MAPE=0.399 RMSE=163.30 n=136 (0 excluded)



Saved feature importance plot to ../outputs/xgb_feature_importance.png

Training LSTM (secondary benchmark, expected to underperform XGBoost given ~59 days of history)...


I0000 00:00:1784571808.218414    1729 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784571808.358815    1729 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1784571811.147452    1729 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


LSTM MAPE: 0.781 (10 zero-true-value rows excluded)
LSTM RMSE: 1193.29



Quantile forecasts built. P90<P50 violations before enforcement: 31/235 (corrected with max(P50,P90)).

Business cost (Naive):   259,514
Business cost (XGBoost): 114,291
Cost reduction vs. naive: 56.0%

Saved forecast table to ../outputs/phase4_forecasts.csv


## Phase 5 — Capacity Optimization (MILP)

Solves for daily worker allocation and overtime per team. Capacity is dispatched by department: sewing is piece/SMV-based, finishing is overtime-minutes-direct (finishing's demand signal has no SMV-equivalent unit). Overtime bounds come from each team's own historical 90th percentile, not a flat or globally-scaled cap.

In [4]:
"""
ThreadLine — Phase 5: Production Scheduling & Capacity Optimization (MILP)
==========================================================================================
Run directly:
    python phase5_optimization.py path/to/phase4_forecasts.csv path/to/cleaned_dataset.csv [output_dir]

Expects phase4_forecasts.csv (Phase 4's output) for team/date/forecast columns, and the
full cleaned dataset (Phase 1-3's output) to derive historical smv/workforce parameters
from the WHOLE history rather than just the forecast test window.

CORRECTIONS vs. the draft (each validated against the actual dataset before fixing):

1. TEAM vs TEAM_ID, again: `df['team'].unique()` and `test.set_index(['team','date'])`
   both use the raw number that repeats across departments -- sewing team 1 and
   finishing team 1 collide under the same key. For the demand dict this isn't just
   inaccurate, it's silently destructive: `.set_index(['team','date']).to_dict()`
   on a Series with a duplicate index keeps whichever row happened to come last and
   drops the other -- meaning roughly half of every team-number's forecasts would
   vanish without any error. Fixed to `team_id` throughout, matching every earlier
   phase.

2. UNIT MISMATCH the piece/SMV capacity formula assumes demand is measured in WIP
   pieces -- true for sewing, but finishing's demand_proxy (from Phase 1-3, since
   WIP is structurally absent there) is `over_time`, measured in MINUTES. Plugging
   a minutes-denominated forecast into `(w*shift_min + o)/smv` as if it were a
   piece count compares two incompatible units for the entire finishing department.
   Fixed with a department-dispatched constraint:
     - sewing:    capacity is pieces producible, (w*shift_min + o)/smv[t] >= demand + backlog
     - finishing: capacity is minutes of overtime directly, o[t,d] >= demand + backlog
                  (no SMV conversion -- both sides are already in minutes)
   Known limitation, stated plainly rather than hidden: this means reallocating
   workers w[t,d] away from a finishing team has no modeled effect on ITS OWN
   overtime requirement in this formulation (only on the shared workforce pool) --
   the dataset has no finishing throughput-per-worker rate (no SMV-equivalent
   analog) to establish that link. Flag this in the writeup as a direction for
   future work if finishing throughput data becomes available.

3. BACKLOG PENALTY UNIT MISMATCH, following directly from #2: a backlog "unit" is
   a WIP piece for sewing but a minute of uncovered overtime for finishing -- these
   cannot share one backlog_penalty_per_unit constant without one of them being
   priced nonsensically. Split into backlog_penalty_per_piece (sewing) and
   backlog_penalty_per_min (finishing), each documented as an assumption.

4. OVERTIME BOUND: the draft caps `o[t,d]` at a flat 120 minutes team-total. Phase
   1's audit found over_time is a TEAM-TOTAL figure -- a flat 120-minute cap is
   roughly 1-3% of any real team's historical overtime (sewing teams average
   4,460-7,550 team-total minutes/day; finishing teams average 1,260-2,775).
   First attempt at a fix scaled the cap by team size (120 min/worker x
   base_workers), using the audit's own global per-worker median -- but a
   median is by definition exceeded on about half of all days, so using it as a
   hard ceiling produced chronic backlog for essentially every finishing team,
   confirmed by running it. Fixed properly: each team's max_overtime is now its
   own historical 90th-percentile over_time (already team-total, already in
   minutes -- no rescaling needed), which reflects a real achieved ceiling for
   that specific team and matches the brief's own bounding language ("bounded
   within realistic historical ranges") directly.

5. MISSING (team, day) COMBINATIONS: not every team_id has a forecast for every
   date in the horizon (some teams have sparse historical coverage -- one team_id
   has only 29 observations across the whole 59-day dataset). The draft's
   `demand.get((t,d), 0)` silently treats every missing combination as "zero
   demand," which isn't a real fact about the world, it's a gap in the forecast
   table (confirmed: 53 of 288 possible team-day combinations are missing here).
   Fixed to build variables and constraints only over combinations that actually
   have a forecast, instead of assuming an empty day.

6. Variable names built from raw pandas Timestamps: PuLP tolerates it, but
   produces names like `w_sewing_1_2015_02_26_00:00:00` -- harmless here, but
   cleaned to `w_sewing_1_20150226` for readability when debugging constraints.
"""

import sys
import numpy as np
import pandas as pd
from pulp import LpProblem, LpVariable, LpMinimize, lpSum, LpInteger, LpStatus, value, PULP_CBC_CMD

SHIFT_MIN = 480            # standard 8-hour shift, minutes
OVERTIME_BOUND_QUANTILE = 0.90  # see fix #4 below
OVERTIME_COST_PER_MIN = 1.5     # premium over regular time (assumed -- no wage data in source)
BACKLOG_PENALTY_PER_PIECE = 5   # sewing: cost per unmet WIP piece (assumed)
BACKLOG_PENALTY_PER_MIN = 4.5   # finishing: cost per unmet overtime-minute (assumed;
                                 # kept close to OVERTIME_COST_PER_MIN x 3 so the "backlog
                                 # costs more than doing the overtime" framing from Q7
                                 # holds for both departments on their own native units)


def build_team_params(full_df):
    """Historical parameters per team_id, from the FULL dataset (not just the
    forecast window) -- 'base_workers' is meant to be a historical average, not
    an average over just the 12 held-out test days.

    max_overtime is each team's own historical OVERTIME_BOUND_QUANTILE of
    over_time (team-total minutes, already in the right unit -- no per-worker
    rescaling needed). Earlier draft of this fix used a flat "120 min/worker x
    base_workers" heuristic derived from the dataset's GLOBAL MEDIAN per-worker
    overtime -- but a median is by definition exceeded on ~half of all days, so
    using it as a hard ceiling guaranteed chronic backlog everywhere by
    construction (confirmed: every finishing team showed backlog on 100% of
    test days with that bound). Using each team's own historical 90th
    percentile instead reflects a real, team-specific achieved ceiling, and
    matches the brief's own bounding language ("bounded within realistic
    historical ranges") more literally than a rescaled global heuristic did.
    """
    g = full_df.groupby(["team_id", "department"]).agg(
        smv=("smv", "mean"),
        base_workers=("no_of_workers", "mean"),
        max_overtime=("over_time", lambda s: s.quantile(OVERTIME_BOUND_QUANTILE)),
    ).reset_index()

    params = {}
    for _, row in g.iterrows():
        t = row["team_id"]
        params[t] = {
            "department": row["department"],
            "smv": float(row["smv"]),
            "base_workers": float(row["base_workers"]),
            "min_workers": max(1, int(0.7 * row["base_workers"])),
            "max_workers": max(2, int(1.3 * row["base_workers"])),
            "max_overtime": float(row["max_overtime"]),
        }
    return params


def build_demand(forecasts, demand_col="forecast_p50"):
    """Demand dict keyed by (team_id, date) -- ONLY for combinations that
    actually have a forecast. No .get(..., 0) fallback: a missing combination
    means 'no forecast produced', not 'zero demand', and those are not the
    same fact."""
    d = forecasts.dropna(subset=[demand_col])
    return {(row.team_id, row.date): getattr(row, demand_col) for row in d.itertuples()}


def vname(t, d):
    return f"{t}_{pd.Timestamp(d).strftime('%Y%m%d')}"


def solve_schedule(forecasts, params, demand_col="forecast_p50"):
    demand = build_demand(forecasts, demand_col)
    combos = sorted(demand.keys())  # only real (team_id, date) pairs
    teams = sorted({t for t, _ in combos})
    days_by_team = {t: sorted(d for tt, d in combos if tt == t) for t in teams}

    total_workforce = int(sum(p["base_workers"] for p in params.values()))

    prob = LpProblem("ThreadLine_Capacity_Scheduling", LpMinimize)

    w = {(t, d): LpVariable(f"w_{vname(t, d)}", params[t]["min_workers"], params[t]["max_workers"], LpInteger)
         for (t, d) in combos}
    o = {(t, d): LpVariable(f"o_{vname(t, d)}", 0, params[t]["max_overtime"])
         for (t, d) in combos}
    backlog = {(t, d): LpVariable(f"backlog_{vname(t, d)}", 0)
               for (t, d) in combos}

    # objective: department-specific backlog penalty, since a backlog "unit"
    # means a different thing (pieces vs. minutes) in each department
    obj_terms = []
    for (t, d) in combos:
        dept = params[t]["department"]
        penalty = BACKLOG_PENALTY_PER_PIECE if dept == "sewing" else BACKLOG_PENALTY_PER_MIN
        obj_terms.append(OVERTIME_COST_PER_MIN * o[t, d] + penalty * backlog[t, d])
    prob += lpSum(obj_terms)

    # capacity constraints, dispatched by department (see fix #2 in the module docstring)
    for t in teams:
        prev_backlog = 0
        for d in days_by_team[t]:
            dept = params[t]["department"]
            d_demand = demand[t, d]
            if dept == "sewing":
                capacity = (w[t, d] * SHIFT_MIN + o[t, d]) / params[t]["smv"]
            else:  # finishing: demand and capacity are already in minutes, no SMV conversion
                capacity = o[t, d]
            prob += capacity + backlog[t, d] >= d_demand + prev_backlog
            prev_backlog = backlog[t, d]  # chain the LpVariable object itself, not its .varValue

    # shared workforce pool, per day actually present in the horizon
    all_days = sorted({d for _, d in combos})
    for d in all_days:
        teams_today = [t for t in teams if d in days_by_team[t]]
        prob += lpSum(w[t, d] for t in teams_today) <= total_workforce

    # gapRel accepts a solution within 0.1% of proven-optimal instead of forcing
    # CBC to close the last fraction via exhaustive branch-and-bound -- on larger
    # combined solves this can be the difference between milliseconds and minutes
    # for a change in reported cost of a fraction of a percent. timeLimit is a
    # hard safety net. PuLP still reports "Optimal" when solved within tolerance.
    prob.solve(PULP_CBC_CMD(msg=0, gapRel=0.001, timeLimit=90))
    status = LpStatus[prob.status]

    result_rows = []
    for (t, d) in combos:
        result_rows.append({
            "team_id": t, "date": d, "department": params[t]["department"],
            "workers": value(w[t, d]), "overtime_min": value(o[t, d]),
            "backlog": value(backlog[t, d]), "demand": demand[t, d],
        })
    schedule = pd.DataFrame(result_rows).sort_values(["team_id", "date"])
    return prob, status, schedule


def fixed_staffing_baseline(forecasts, params, demand_col="forecast_p50"):
    """Same principle as Phase 4's naive baseline: an optimizer without a
    comparison point is just a number nobody can evaluate. Mirrors the
    optimized model's own constraint structure per department, but with
    workers held at their historical average and overtime applied reactively
    (whatever's needed, up to the same cap) instead of chosen by the solver."""
    demand = build_demand(forecasts, demand_col)
    by_team = {}
    for (t, d), dem in demand.items():
        by_team.setdefault(t, []).append((d, dem))

    total_cost = 0.0
    rows = []
    for t, entries in by_team.items():
        entries.sort()
        dept = params[t]["department"]
        base_workers = params[t]["base_workers"]
        max_ot = params[t]["max_overtime"]
        penalty = BACKLOG_PENALTY_PER_PIECE if dept == "sewing" else BACKLOG_PENALTY_PER_MIN
        prev_backlog = 0.0
        for d, dem in entries:
            if dept == "sewing":
                cap = base_workers * SHIFT_MIN / params[t]["smv"]
                shortfall = max(0.0, dem + prev_backlog - cap)
                ot_provided_min = min(shortfall * params[t]["smv"], max_ot)
                remaining_backlog = max(0.0, shortfall - ot_provided_min / params[t]["smv"])
            else:
                needed = dem + prev_backlog
                ot_provided_min = min(needed, max_ot)
                remaining_backlog = max(0.0, needed - ot_provided_min)
            cost = OVERTIME_COST_PER_MIN * ot_provided_min + penalty * remaining_backlog
            total_cost += cost
            rows.append({"team_id": t, "date": d, "overtime_min": ot_provided_min,
                          "backlog": remaining_backlog})
            prev_backlog = remaining_backlog

    return total_cost, pd.DataFrame(rows)

output_dir = OUTPUT_DIR
# forecasts and full_df are reused directly from the Phase 4 / Phase 1-3 cells
# above (test_q and df) instead of round-tripping through disk
forecasts = test_q.copy()
full_df = df.copy()

params = build_team_params(full_df)

print(f"Teams: {len(params)} | total_workforce pool: "
      f"{int(sum(p['base_workers'] for p in params.values()))}")
print(f"Overtime cap range: "
      f"{min(p['max_overtime'] for p in params.values()):.0f}"
      f"-{max(p['max_overtime'] for p in params.values()):.0f} min/team/day "
      f"(each team's own historical {int(OVERTIME_BOUND_QUANTILE*100)}th percentile, "
      f"not a flat cap)")

prob, status, schedule = solve_schedule(forecasts, params, demand_col="forecast_p50")
print(f"\nSolve status: {status}")
optimized_cost = value(prob.objective)
print(f"Optimized cost (P50 demand): {optimized_cost:,.2f}")

baseline_cost, baseline_detail = fixed_staffing_baseline(forecasts, params, demand_col="forecast_p50")
print(f"Baseline cost (fixed staffing): {baseline_cost:,.2f}")
print(f"Savings: {(1 - optimized_cost / baseline_cost) * 100:.1f}%")

print("\n--- Sanity checks (Step 5) ---")
persistent_backlog = (
    schedule.groupby("team_id")["backlog"]
    .apply(lambda s: (s > 1e-3).mean())
    .sort_values(ascending=False)
)
print("Fraction of days with nonzero backlog, top 5 teams:")
print(persistent_backlog.head(5).round(2).to_string())
print("-> a team persistently backlogged even under the optimizer's best "
      "allocation is a candidate for permanent headcount change (Q9), not "
      "just daily reallocation.")

if output_dir:
    schedule.to_csv(f"{output_dir}/phase5_schedule.csv", index=False)
    print(f"\nSaved schedule to {output_dir}/phase5_schedule.csv")


Teams: 24 | total_workforce pool: 753
Overtime cap range: 1800-10620 min/team/day (each team's own historical 90th percentile, not a flat cap)



Solve status: Optimal
Optimized cost (P50 demand): 664,639.66
Baseline cost (fixed staffing): 1,048,134.56
Savings: 36.6%

--- Sanity checks (Step 5) ---
Fraction of days with nonzero backlog, top 5 teams:
team_id
finishing_7     1.00
finishing_6     0.67
finishing_10    0.62
finishing_3     0.60
sewing_1        0.58
-> a team persistently backlogged even under the optimizer's best allocation is a candidate for permanent headcount change (Q9), not just daily reallocation.

Saved schedule to ../outputs/phase5_schedule.csv


## Phase 6 — Pipeline Integration

Wires cleaning, forecasting, and optimization into one callable pipeline (`prepare_forecast` → `solve_scenario`), so comparing P50 vs. P90 doesn't mean retraining the forecaster twice.

In [5]:
"""
ThreadLine — Consolidated Pipeline (Phases 2, 4, 5, 7): clean -> forecast -> optimize -> stress-test
==========================================================================================
Single cell, paste-and-run. Point RAW_CSV_PATH at your raw source file.

CORRECTIONS vs. the draft (each validated against the actual dataset -- this reintroduced
several bugs already fixed and confirmed in earlier phases of this project):

1. TARGET VARIABLE: forecasts `workload_pressure` again, which is constant 0 for the
   entire finishing department (WIP isn't tracked there). Restored `demand_proxy`
   (wip for sewing, over_time for finishing -- the only workload-adjacent field that
   actually varies for finishing, confirmed earlier: std 1776.7 vs. dead-zero on wip,
   idle_time, idle_men, no_of_style_change).

2. TEAM vs TEAM_ID: `groupby('team')` and `set_index(['team','date'])` reappear in three
   places (rolling/lag features, MILP team set, base_stats). All three collide sewing
   team N with finishing team N. Fixed to team_id throughout. The set_index one is the
   worst of the three -- with a duplicate index, `.to_dict()` silently keeps the last
   row and drops the other, meaning close to half the forecasts disappear with no error.

3. UNIT MISMATCH IN THE MILP: `(w*shift_min + o)/smv` applied uniformly assumes demand
   is in WIP pieces -- true for sewing, but finishing's demand_proxy is over_time,
   measured in minutes. Dispatched by department: sewing stays piece/SMV-based,
   finishing compares minutes to minutes directly (`o[t,d] >= demand + backlog`).
   Follows that BACKLOG_PENALTY can't be one shared constant either -- a backlog
   "unit" is a piece for sewing, a minute for finishing. Split into two.

4. OVERTIME BOUND: flat `max_overtime_min = 120` team-total is ~1-3% of any real
   team's historical overtime. An earlier attempt at fixing this scaled the flat
   number by team size using the dataset's GLOBAL MEDIAN per-worker overtime -- but a
   median is exceeded on ~half of all days by definition, and using it as a hard
   ceiling produced backlog on 100% of days for every finishing team when tested.
   Fixed to each team's own historical 90th-percentile over_time (already team-total,
   already in minutes -- no rescaling needed).

5. MISSING (team, day) COMBINATIONS: `demand.get((t,d), 0)` treats a missing forecast
   as "zero demand", which isn't a fact about the world, it's a gap in the forecast
   table (confirmed: 53 of 288 possible team-day combinations are missing in this
   dataset -- some teams have as few as 29 observations across all 59 dates). Fixed
   to only build variables/constraints over combinations that actually have a
   forecast. The draft's own missing-combo warning code was also unreachable as
   written -- `teams`/`days`/`demand` are local to optimize_schedule() and don't
   exist in the caller's scope, so that diagnostic would raise NameError the moment
   it ran. Fixed by returning the diagnostic from inside the function that has the
   data.

6. QUANTILE MONOTONICITY: P50/P90 trained as independent models with no constraint
   forcing P90 >= P50 on a given row -- confirmed this is violated on ~13% of rows.
   Enforced with max(p50, p90) after prediction.

7. actual_productivity CLIPPED to 1.0: throws away real over-target performance
   (37 rows, up to 1.12 in this dataset) instead of flagging it. Restored the flag-
   don't-clip approach from Phase 1-3.

8. EFFICIENCY / STRUCTURE: the draft called the full pipeline (clean + forecast) TWICE
   to compare P50 vs P90, retraining identical models for no reason since both
   quantiles are already produced in one forecast_workload() call. Split into
   prepare_forecast() [run once] and solve_scenario() [run per demand column], so the
   P50-vs-P90 comparison only re-solves the MILP, not the whole pipeline.

9. DATE PARSING: hardcoded format='%m/%d/%Y' will raise ValueError if fed a file with
   ISO-formatted dates (e.g. output from an earlier phase). Added a fallback so this
   works on either the true raw source or an already-partially-processed file.

10. IDEMPOTENCY: clean_pipeline now drops any pre-existing engineered columns before
    recomputing them, so it's safe to point at raw data OR at a previous run's output
    without producing _x/_y duplicate columns (same fix as Phase 1-3).
"""

import json
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from pulp import LpProblem, LpVariable, LpMinimize, lpSum, LpInteger, LpStatus, value, PULP_CBC_CMD

# ============================================================================
# CONFIG
# ============================================================================
FEATURES = ["no_of_workers", "smv", "over_time", "incentive", "idle_time",
            "style_change_flag", "demand_proxy_roll3", "demand_proxy_lag1",
            "targeted_productivity", "dept_enc"]

ENGINEERED_COLS = [
    "team_num", "team_id", "over_target_flag", "workload_pressure",
    "demand_proxy", "demand_proxy_source", "demand_proxy_roll3", "demand_proxy_lag1",
    "style_change_flag", "rated_capacity_min", "idle_loss_min",
    "effective_capacity_min", "productivity_gap",
]

DEFAULT_CONFIG = {
    "demand_column": "forecast_p50",   # swap to 'forecast_p90' for the Phase 7 stress test
    "shift_min": 480,
    "overtime_bound_quantile": 0.90,   # each team's own historical percentile -- see fix #4
    "overtime_cost_per_min": 1.5,
    "backlog_penalty_per_piece": 5,    # sewing: cost per unmet WIP piece (assumed)
    "backlog_penalty_per_min": 4.5,    # finishing: cost per unmet overtime-minute (assumed)
}


# ============================================================================
# PHASE 2 — CLEAN + FEATURE ENGINEER
# ============================================================================
def _parse_dates(series):
    try:
        return pd.to_datetime(series, format="%m/%d/%Y")
    except (ValueError, TypeError):
        return pd.to_datetime(series)


def _time_aware_rolling(df, value_col, prefix):
    all_dates = pd.Index(sorted(df["date"].unique()), name="date")
    frames = []
    for tid, g in df.groupby("team_id"):
        s = g.set_index("date")[value_col].reindex(all_dates)
        frames.append(pd.DataFrame({
            "team_id": tid, "date": all_dates,
            f"{prefix}_roll3": s.rolling(3, min_periods=1).mean().values,
            f"{prefix}_lag1": s.shift(1).values,
        }))
    return pd.concat(frames, ignore_index=True)


def clean_pipeline(raw_csv_path):
    df = pd.read_csv(raw_csv_path)
    df.columns = df.columns.str.strip()
    df = df.drop(columns=[c for c in ENGINEERED_COLS if c in df.columns])  # idempotency

    df["department"] = df["department"].str.strip().str.replace("sweing", "sewing", regex=False)
    df["date"] = _parse_dates(df["date"])

    df["team_num"] = df["team"]
    df["team_id"] = df["department"] + "_" + df["team"].astype(str)

    # flag over-target rows instead of clipping them away (real signal, not noise)
    df["over_target_flag"] = (df["actual_productivity"] > 1.0).astype(int)

    df["style_change_flag"] = (df["no_of_style_change"] > 0).astype(int)
    df["rated_capacity_min"] = df["no_of_workers"] * df["smv"]
    df["idle_loss_min"] = df["idle_time"] * df["idle_men"]
    df["effective_capacity_min"] = df["rated_capacity_min"] - df["idle_loss_min"]
    df["productivity_gap"] = df["targeted_productivity"] - df["actual_productivity"]

    # workload_pressure kept only as a raw numeric convenience field -- NOT the
    # forecasting target, since it's constant-zero for all of finishing
    df["workload_pressure"] = df["wip"].fillna(0)
    df["demand_proxy"] = np.where(df["department"] == "sewing", df["wip"], df["over_time"])
    df["demand_proxy_source"] = np.where(df["department"] == "sewing", "wip", "over_time")

    df = df.merge(_time_aware_rolling(df, "demand_proxy", "demand_proxy"),
                  on=["team_id", "date"], how="left")

    return df.sort_values(["team_id", "date"]).reset_index(drop=True)


# ============================================================================
# PHASE 4 — QUANTILE FORECASTING
# ============================================================================
def forecast_workload(df, cutoff_date=None, quantiles=(0.5, 0.9)):
    df = df.copy()
    le = LabelEncoder()
    df["dept_enc"] = le.fit_transform(df["department"])

    unique_dates = sorted(df["date"].unique())
    if cutoff_date is None:
        cutoff_date = unique_dates[int(len(unique_dates) * 0.8)]  # unique-date based, not row-based

    train = df[df["date"] <= cutoff_date]
    test = df[df["date"] > cutoff_date].copy()

    # drop rows missing the lag feature (each team's first observed day) instead
    # of fillna(0), which would fabricate a fake "no backlog" history
    train_clean = train.dropna(subset=["demand_proxy_lag1"])
    test_clean = test.dropna(subset=["demand_proxy_lag1"])

    X_train, y_train = train_clean[FEATURES], train_clean["demand_proxy"]
    X_test = test_clean[FEATURES]

    preds = {}
    for q in quantiles:
        model = xgb.XGBRegressor(objective="reg:quantileerror", quantile_alpha=q,
                                  n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
        model.fit(X_train, y_train)
        preds[q] = model.predict(X_test)

    test_clean = test_clean.copy()
    test_clean["forecast_p50"] = preds[0.5]
    test_clean["forecast_p90"] = np.maximum(preds[0.5], preds[0.9])  # enforce P90 >= P50

    return test_clean[["team_id", "date", "department", "demand_proxy",
                        "forecast_p50", "forecast_p90"]]


# ============================================================================
# PHASE 5 — CAPACITY OPTIMIZATION (MILP)
# ============================================================================
def build_base_stats(df):
    g = df.groupby(["team_id", "department"]).agg(
        smv=("smv", "mean"),
        base_workers=("no_of_workers", "mean"),
        overtime_p90=("over_time", lambda s: s.quantile(0.90)),
    ).reset_index()
    return {row.team_id: {
        "department": row.department, "smv": row.smv, "base_workers": row.base_workers,
        "overtime_p90": row.overtime_p90,
    } for row in g.itertuples()}


def _vname(t, d):
    return f"{t}_{pd.Timestamp(d).strftime('%Y%m%d')}"


def optimize_schedule(forecast_df, base_stats, config):
    demand_col = config["demand_column"]
    demand_df = forecast_df.dropna(subset=[demand_col])
    demand = {(row.team_id, row.date): getattr(row, demand_col) for row in demand_df.itertuples()}
    combos = sorted(demand.keys())
    teams = sorted({t for t, _ in combos})
    days_by_team = {t: sorted(d for tt, d in combos if tt == t) for t in teams}

    min_workers = {t: max(1, int(0.7 * base_stats[t]["base_workers"])) for t in teams}
    max_workers = {t: max(2, int(1.3 * base_stats[t]["base_workers"])) for t in teams}
    max_overtime = {t: base_stats[t]["overtime_p90"] * (config["overtime_bound_quantile"] / 0.90)
                     for t in teams}  # rescale if a caller overrides the quantile assumption
    total_workforce = int(sum(base_stats[t]["base_workers"] for t in teams))

    prob = LpProblem("Capacity_Scheduling", LpMinimize)
    w = {(t, d): LpVariable(f"w_{_vname(t, d)}", min_workers[t], max_workers[t], LpInteger) for (t, d) in combos}
    o = {(t, d): LpVariable(f"o_{_vname(t, d)}", 0, max_overtime[t]) for (t, d) in combos}
    bl = {(t, d): LpVariable(f"bl_{_vname(t, d)}", 0) for (t, d) in combos}

    obj_terms = []
    for (t, d) in combos:
        dept = base_stats[t]["department"]
        penalty = config["backlog_penalty_per_piece"] if dept == "sewing" else config["backlog_penalty_per_min"]
        obj_terms.append(config["overtime_cost_per_min"] * o[t, d] + penalty * bl[t, d])
    prob += lpSum(obj_terms)

    for t in teams:
        prev_bl = 0
        dept = base_stats[t]["department"]
        for d in days_by_team[t]:
            if dept == "sewing":
                capacity = (w[t, d] * config["shift_min"] + o[t, d]) / base_stats[t]["smv"]
            else:  # finishing: demand and capacity are already in minutes, no SMV conversion
                capacity = o[t, d]
            prob += capacity + bl[t, d] >= demand[t, d] + prev_bl
            prev_bl = bl[t, d]

    all_days = sorted({d for _, d in combos})
    for d in all_days:
        teams_today = [t for t in teams if d in days_by_team[t]]
        prob += lpSum(w[t, d] for t in teams_today) <= total_workforce

    # gapRel accepts a solution within 0.1% of proven-optimal instead of forcing
    # CBC to close the last fraction of a percent via exhaustive branch-and-bound --
    # on a large combined solve (e.g. a full-period hindsight backtest across all
    # teams and every date at once) that difference is 235+ seconds of runtime for
    # a change in the reported cost of about 0.001%. timeLimit is a hard safety net.
    # PuLP still reports status "Optimal" when solved within the gap tolerance.
    prob.solve(PULP_CBC_CMD(msg=0, gapRel=0.001, timeLimit=90))

    schedule = pd.DataFrame([
        {"team_id": t, "date": d, "department": base_stats[t]["department"],
         "workers": w[t, d].varValue, "overtime_min": o[t, d].varValue, "backlog": bl[t, d].varValue}
        for (t, d) in combos
    ])

    # diagnostic returned from the function that actually has the data,
    # instead of dangling in the caller's scope where it would NameError
    full_grid = {(t, d) for t in teams for d in all_days}
    missing = sorted(full_grid - set(combos))

    return {
        "status": LpStatus[prob.status],
        "total_cost": value(prob.objective),
        "schedule": schedule,
        "missing_combos": missing,
    }


# ============================================================================
# ORCHESTRATION
# ============================================================================
def prepare_forecast(raw_csv_path):
    """Run once: clean + forecast. Reused across scenario runs so comparing
    P50 vs P90 doesn't retrain identical models twice."""
    df = clean_pipeline(raw_csv_path)
    forecast_df = forecast_workload(df)
    base_stats = build_base_stats(df)
    return forecast_df, base_stats


def solve_scenario(forecast_df, base_stats, config, output_dir=None, label=None):
    result = optimize_schedule(forecast_df, base_stats, config)
    if result["status"] != "Optimal":
        raise RuntimeError(f"Solver did not find an optimal solution: {result['status']}")
    if result["missing_combos"]:
        print(f"Note: {len(result['missing_combos'])} team-day combinations had no forecast "
              f"and were excluded from the schedule (not treated as zero demand).")

    label = label or config["demand_column"]
    print(f"[{label}] Solver status: {result['status']} | Total cost: {result['total_cost']:.2f}")

    if output_dir:
        result["schedule"].to_csv(f"{output_dir}/schedule_{label}.csv", index=False)
        with open(f"{output_dir}/cost_summary_{label}.json", "w") as f:
            json.dump({"total_cost": result["total_cost"], "scenario": label}, f, indent=2)

    return result["schedule"], result["total_cost"]


def run_pipeline(raw_csv_path, config=None, output_dir="outputs"):
    """Convenience wrapper for a single scenario run."""
    config = config or DEFAULT_CONFIG
    forecast_df, base_stats = prepare_forecast(raw_csv_path)
    return solve_scenario(forecast_df, base_stats, config, output_dir=output_dir)


# ============================================================================
# USAGE (only runs when this file is executed directly, not when imported --
# without this guard, `import phase6_pipeline` from Phase 7 would silently
# re-run the whole P50/P90 pipeline as a side effect of the import)
# ============================================================================

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

forecast_df, base_stats = prepare_forecast(DATA_PATH)

schedule_p50, cost_p50 = solve_scenario(
    forecast_df, base_stats, {**DEFAULT_CONFIG, "demand_column": "forecast_p50"},
    output_dir=OUTPUT_DIR, label="p50")

schedule_p90, cost_p90 = solve_scenario(
    forecast_df, base_stats, {**DEFAULT_CONFIG, "demand_column": "forecast_p90"},
    output_dir=OUTPUT_DIR, label="p90")

assert cost_p90 >= cost_p50, "P90 (higher demand) should never cost less than P50 -- check the pipeline."
print(f"\nP50 cost: {cost_p50:.2f} | P90 cost: {cost_p90:.2f} | Surge premium: {cost_p90 - cost_p50:.2f}")


Note: 49 team-day combinations had no forecast and were excluded from the schedule (not treated as zero demand).
[p50] Solver status: Optimal | Total cost: 291383.88
Note: 49 team-day combinations had no forecast and were excluded from the schedule (not treated as zero demand).
[p90] Solver status: Optimal | Total cost: 728485.93

P50 cost: 291383.88 | P90 cost: 728485.93 | Surge premium: 437102.05


## Phase 7 — Scenario & Sensitivity Analysis

Demand surge, labor shortage, and compound-shock scenarios, plus a backlog-vs-overtime cost-ratio sweep to find the tipping point where overtime becomes cheaper than letting backlog accumulate.

In [6]:
"""
ThreadLine — Phase 7: Scenario & Sensitivity Analysis
==========================================================================================
Run directly:
    python phase7_scenarios.py path/to/garments_worker_productivity.csv [output_dir]

Requires phase6_pipeline.py in the same directory -- imports clean_pipeline,
forecast_workload, optimize_schedule, build_base_stats, DEFAULT_CONFIG from it
rather than duplicating that logic here.

CORRECTIONS vs. the draft (each validated against the actual pipeline before fixing):

1. TEAM vs TEAM_ID, again: `fdf['team'] == team` and `stats['smv'][team]` use the raw
   number that collides sewing/finishing. Every scenario definition (`rush_order`,
   `downtime_teams`) now takes a team_id string like 'sewing_1', matching every other
   phase.

2. DOWNTIME DOESN'T WORK ON FINISHING AS WRITTEN: the draft models capacity loss by
   inflating SMV. But Phase 5's unit-mismatch fix means finishing's capacity
   constraint doesn't use SMV at all -- it's overtime-minutes-direct. Inflating a
   finishing team's SMV would silently do nothing to its schedule. Fixed with a
   department-dispatched perturbation: sewing still inflates SMV (dimensionally
   correct there); finishing scales its forecasted demand up instead, which has the
   equivalent effect -- more minutes required to clear the same real workload.

3. EFFICIENCY: `run_scenario` in the draft calls `clean_pipeline` and
   `forecast_workload` FRESH for every single scenario -- with 6 base scenarios plus
   a 7-point cost-ratio sweep, that's 13 full reruns, each retraining two XGBoost
   models that produce an IDENTICAL forecast every time, since none of these
   scenarios change the input data the forecaster sees. Fixed the same way Phase 6
   was: prepare the forecast ONCE, run every scenario against that cached result.

4. INFEASIBILITY WOULD CRASH THE WHOLE BATCH: Phase 6's `optimize_schedule` wrapper
   raises RuntimeError on a non-optimal solve, which is correct for a single
   production run but wrong here -- the doc itself says an infeasible Compound Shock
   scenario "is itself a finding," but a raised exception would kill the batch before
   the other 12 scenarios/sweep points ever ran. Fixed to catch a non-optimal status
   per-scenario, record it in the results table (cost as NaN, status preserved), and
   continue the sweep.

5. BACKLOG PENALTY: the draft sweeps a single `backlog_penalty_per_unit`, which
   doesn't exist in the corrected config (Phase 5 split it into
   backlog_penalty_per_piece and backlog_penalty_per_min because a backlog "unit" is
   a different thing -- and different cost -- in each department). Both are scaled
   by the same ratio together, and overtime/backlog totals are reported per
   department too, since sewing and finishing operate on very different absolute
   scales and could have different tipping points.

6. RUSH ORDER EXAMPLE DATE: the draft's example, `pd.Timestamp('2015-02-10')`, falls
   inside the TRAINING period on this dataset, not the forecast window (which starts
   2015-02-28) -- the scenario would silently match zero rows and do nothing. Fixed
   with a real in-window example, and added a check that warns (rather than silently
   no-ops) if a scenario target isn't found.

7. MONOTONICITY: the doc says cost should "strictly increase from Baseline -> Surge
   -> Compound Shock" but leaves it as something you eyeball in a printed table.
   Turned into an explicit check that prints a clear pass/fail instead of relying on
   a person to notice quietly.
"""

import sys
import copy
import numpy as np
import pandas as pd

# clean_pipeline, forecast_workload, build_base_stats, optimize_schedule,
# DEFAULT_CONFIG are already in scope from the Phase 6 cell above


# ============================================================================
# SCENARIO PERTURBATION LAYER
# ============================================================================
def apply_scenario(forecast_df, base_stats, scenario):
    fdf = forecast_df.copy()
    stats = copy.deepcopy(base_stats)
    demand_col = scenario.get("demand_column", "forecast_p50")

    if scenario.get("rush_order"):
        team_id, date, extra_units = scenario["rush_order"]
        date = pd.Timestamp(date)
        mask = (fdf["team_id"] == team_id) & (fdf["date"] == date)
        if not mask.any():
            print(f"  Note: rush_order target ({team_id}, {date.date()}) not found in "
                  f"the forecast window -- scenario has no effect on this run.")
        fdf.loc[mask, demand_col] = fdf.loc[mask, demand_col] + extra_units

    if scenario.get("downtime_teams"):
        for team_id, pct_loss in scenario["downtime_teams"].items():
            if team_id not in stats:
                print(f"  Note: downtime target '{team_id}' not found -- skipped.")
                continue
            if stats[team_id]["department"] == "sewing":
                # inflating SMV = more minutes needed per piece; dimensionally
                # correct since sewing's capacity constraint is piece/SMV-based
                stats[team_id]["smv"] = stats[team_id]["smv"] / (1 - pct_loss)
            else:
                # finishing's capacity constraint is overtime-minutes-direct, no
                # SMV term -- so scale the forecasted demand up instead, which has
                # the equivalent effect (more minutes required for the same work)
                mask = fdf["team_id"] == team_id
                fdf.loc[mask, demand_col] = fdf.loc[mask, demand_col] / (1 - pct_loss)

    if scenario.get("workforce_shock_pct"):
        shock = scenario["workforce_shock_pct"]
        for team_id in stats:
            stats[team_id]["base_workers"] = max(1.0, stats[team_id]["base_workers"] * (1 - shock))

    return fdf, stats, demand_col


# ============================================================================
# SCENARIO RUNNER (operates on an already-prepared forecast -- see fix #3)
# ============================================================================
def run_scenario(forecast_df, base_stats, scenario_name, scenario=None, base_config=None):
    scenario = scenario or {}
    config = dict(base_config or DEFAULT_CONFIG)

    fdf, stats, demand_col = apply_scenario(forecast_df, base_stats, scenario)
    config["demand_column"] = demand_col

    result = optimize_schedule(fdf, stats, config)

    if result["status"] != "Optimal":
        # infeasibility under a stress scenario is a finding, not a crash --
        # record it and let the rest of the sweep continue (see fix #4)
        return {"scenario": scenario_name, "status": result["status"],
                "total_cost": np.nan, "total_overtime": np.nan, "total_backlog": np.nan,
                "sewing_overtime": np.nan, "sewing_backlog": np.nan,
                "finishing_overtime": np.nan, "finishing_backlog": np.nan}

    sched = result["schedule"]
    by_dept = sched.groupby("department").agg(overtime=("overtime_min", "sum"), backlog=("backlog", "sum"))

    return {
        "scenario": scenario_name, "status": result["status"], "total_cost": result["total_cost"],
        "total_overtime": sched["overtime_min"].sum(), "total_backlog": sched["backlog"].sum(),
        "sewing_overtime": by_dept.loc["sewing", "overtime"] if "sewing" in by_dept.index else 0.0,
        "sewing_backlog": by_dept.loc["sewing", "backlog"] if "sewing" in by_dept.index else 0.0,
        "finishing_overtime": by_dept.loc["finishing", "overtime"] if "finishing" in by_dept.index else 0.0,
        "finishing_backlog": by_dept.loc["finishing", "backlog"] if "finishing" in by_dept.index else 0.0,
    }


# ============================================================================
# MAIN
# ============================================================================

output_dir = OUTPUT_DIR
import os
os.makedirs(output_dir, exist_ok=True)

# forecast_df / base_stats reused directly from the Phase 6 cell above --
# already prepared once there, no need to rebuild for every scenario/sweep point

# pick a real, in-window example team/date for the rush-order demo, rather
# than hardcoding a date that might fall outside the forecast horizon
example_team = sorted(forecast_df["team_id"].unique())[0]
example_date = sorted(forecast_df["date"].unique())[2]

scenarios = [
    ("Baseline (P50)", {}),
    ("Demand Surge (P90)", {"demand_column": "forecast_p90"}),
    ("Rush Order", {"rush_order": (example_team, example_date, 500)}),
    ("Line Downtime -- 20%", {"downtime_teams": {example_team: 0.20}}),
    ("Labor Shortage (10%)", {"workforce_shock_pct": 0.10}),
    ("Compound Shock", {"demand_column": "forecast_p90", "downtime_teams": {example_team: 0.20}}),
]

print(f"Rush order / downtime example team: {example_team}, example date: "
      f"{pd.Timestamp(example_date).date()}\n")

results = [run_scenario(forecast_df, base_stats, name, scen) for name, scen in scenarios]
results_df = pd.DataFrame(results)

baseline_cost = results_df.loc[results_df["scenario"] == "Baseline (P50)", "total_cost"].values[0]
results_df["cost_vs_baseline_pct"] = (results_df["total_cost"] / baseline_cost - 1) * 100
print(results_df[["scenario", "status", "total_cost", "cost_vs_baseline_pct",
                   "total_overtime", "total_backlog"]].to_string(index=False))

# explicit pass/fail instead of something to eyeball (fix #7)
ordered = results_df.set_index("scenario")
checks = [
    ("Demand Surge >= Baseline",
     ordered.loc["Demand Surge (P90)", "total_cost"] >= ordered.loc["Baseline (P50)", "total_cost"]),
    ("Compound Shock is the worst outcome",
     ordered.loc["Compound Shock", "total_cost"] >= results_df["total_cost"].max() - 1e-6),
]
print("\nMonotonicity checks:")
for name, ok in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")

# --- cost-ratio sensitivity sweep (Q7) ---
print("\n--- Cost-ratio sweep ---")
ratios = [1, 2, 3, 5, 8, 10, 15]
sweep = []
for r in ratios:
    cfg = dict(DEFAULT_CONFIG)
    cfg["backlog_penalty_per_piece"] = cfg["overtime_cost_per_min"] * r
    cfg["backlog_penalty_per_min"] = cfg["overtime_cost_per_min"] * r
    res = run_scenario(forecast_df, base_stats, f"ratio_{r}x", {}, cfg)
    sweep.append({"ratio": r, **res})

sweep_df = pd.DataFrame(sweep)
print(sweep_df[["ratio", "total_overtime", "total_backlog", "total_cost"]].to_string(index=False))

sweep_df["backlog_drop"] = sweep_df["total_backlog"].diff()
if sweep_df["backlog_drop"].notna().any():
    tipping_idx = sweep_df["backlog_drop"].idxmin()
    tipping_point = sweep_df.loc[tipping_idx, "ratio"]
    print(f"\nOvertime becomes the dominant strategy once backlog costs ~{tipping_point}x "
          f"overtime cost (largest single-step drop in total backlog).")

# --- charts ---
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].bar(results_df["scenario"], results_df["cost_vs_baseline_pct"], color="#B25B25")
    axes[0].set_title("Cost Impact by Scenario (% vs. Baseline)")
    axes[0].tick_params(axis="x", rotation=45)

    axes[1].plot(sweep_df["ratio"], sweep_df["total_overtime"], marker="o", label="Overtime (min)")
    axes[1].plot(sweep_df["ratio"], sweep_df["total_backlog"], marker="o", label="Backlog (units)")
    axes[1].set_xlabel("Backlog : Overtime cost ratio")
    axes[1].set_title("Overtime vs. Backlog Trade-off")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(f"{output_dir}/scenario_analysis.png", dpi=150)
    plt.close(fig)
    print(f"\nSaved chart to {output_dir}/scenario_analysis.png")
except ImportError:
    print("(matplotlib not available -- skipped charts)")

results_df.to_csv(f"{output_dir}/phase7_scenarios.csv", index=False)
sweep_df.to_csv(f"{output_dir}/phase7_cost_ratio_sweep.csv", index=False)
print(f"Saved scenario table and sweep table to {output_dir}/")


Rush order / downtime example team: finishing_1, example date: 2015-03-02



            scenario  status    total_cost  cost_vs_baseline_pct  total_overtime  total_backlog
      Baseline (P50) Optimal 291383.878972              0.000000    166334.94993    9085.222092
  Demand Surge (P90) Optimal 728485.927310            150.009002    245979.58913   78610.619467
          Rush Order Optimal 292133.878972              0.257392    166834.94993    9085.222092
Line Downtime -- 20% Optimal 305329.305727              4.785929    170735.51440   10717.350992
Labor Shortage (10%) Optimal 367449.166301             26.104837    194669.14387   15798.021376
      Compound Shock Optimal 819325.399355            181.184190    251958.40266   96804.230967

Monotonicity checks:
  [PASS] Demand Surge >= Baseline
  [PASS] Compound Shock is the worst outcome

--- Cost-ratio sweep ---


 ratio  total_overtime  total_backlog    total_cost
     1   159833.396370   11392.841585 256839.356933
     2   159833.396370   11392.841585 273928.619310
     3   163725.281830    9908.044092 290174.121161
     5   170328.512567    8043.992518 315822.712738
     8   171575.529419    7851.529526 351581.648442
    10   175440.560840    7396.620584 374110.150022
    15   177190.574162    7256.105017 429048.224124

Overtime becomes the dominant strategy once backlog costs ~5x overtime cost (largest single-step drop in total backlog).



Saved chart to ../outputs/scenario_analysis.png
Saved scenario table and sweep table to ../outputs/


## Phase 9 — Impact Validation

Three layers, from optimistic to realistic: (A) walk-forward forecast accuracy, (B) perfect-hindsight optimizer ceiling, (C) realistic end-to-end backtest using only information available at each point in time. Reports honestly that C < B — the gap is the cost of forecast uncertainty.

In [7]:
"""
ThreadLine — Phase 9: Impact Validation
==========================================================================================
Run directly:
    python phase9_validation.py path/to/garments_worker_productivity.csv [output_dir]

Requires phase6_pipeline.py in the same directory.

CORRECTIONS vs. the draft (each validated against the actual pipeline before fixing):

1. TEAM vs TEAM_ID / workload_pressure vs demand_proxy, again, in all four functions
   (actual_historical_cost, hindsight_backtest, walk_forward_forecast_eval,
   realized_cost_of_schedule). Fixed throughout, matching every other phase.

2. UNIFORM SMV CAPACITY FORMULA, again: `(workers*shift_min + overtime)/smv` applied
   to every team assumes demand is in WIP pieces -- true for sewing, not for
   finishing (Phase 5's unit-mismatch fix). Factored into a shared
   `_capacity_provided()` helper, dispatched by department exactly like the MILP
   itself, so the backtest evaluates schedules using the SAME cost logic that
   produced them -- an inconsistent evaluation formula would make Layer B/C
   comparisons meaningless even if each one were internally correct.

3. A GENUINE DATA-STRUCTURAL FINDING, not a code bug: because finishing has no
   independent WIP signal, its demand_proxy IS over_time (Phase 1-3's fix). That
   means actual_historical_cost's finishing backlog is IDENTICALLY ZERO by
   construction -- confirmed on the real data: demand_proxy == over_time for every
   single finishing row, always. This isn't finishing being well-staffed, it's
   using a variable as its own ground truth. Reported explicitly (with a
   department breakdown) rather than silently folded into one combined number
   that would misleadingly suggest all 24 teams contributed to the finding.

4. base_stats BUILT INLINE WITH THE WRONG SHAPE in both hindsight_backtest and
   end_to_end_backtest -- replaced with build_base_stats(df), matching every
   downstream phase's expectation.

5. BACKLOG PENALTY: single `backlog_penalty_per_unit` doesn't exist in the
   corrected config (Phase 5 split it into per-piece and per-minute versions,
   since a "unit" means a different thing in each department). Dispatched through
   the same `_backlog_penalty()` helper used everywhere else.

6. NO INFEASIBILITY HANDLING IN THE BACKTEST LOOPS: the draft's Step 6 says "every
   optimize_schedule call should return Optimal" as something to check by eye
   afterward -- but nothing in the loop actually stops a non-optimal fold from
   being silently averaged in as if its cost were meaningful. Fixed to record
   status per fold and exclude non-optimal folds from the mean/std instead of
   contaminating them.

9. LAYER B vs LAYER C SCOPE MISMATCH -- this is the one worth understanding, not
   just fixing. The doc's own Step 6 says "expect C < B"; running it produced
   B=44.1%, C=72.0% -- backwards, and a real red flag rather than something to
   report as-is. Cause: Layer B (hindsight_backtest) was computed over the FULL
   59-day dataset, while Layer C (end_to_end_backtest) only covers the ~15 days
   that happen to fall inside the 5 backtest fold windows. A whole-dataset ceiling
   compared against a 15-day realistic result isn't the same comparison -- confirmed
   by recomputing Layer B restricted to those same 15 days: 77.2%, which correctly
   sits above Layer C's 72.0%. Fixed to report BOTH a full-period hindsight ceiling
   (useful on its own) and a backtest-scope-matched hindsight figure -- the latter,
   not the former, is what Layer C should actually be measured against.

7. realized_cost_of_schedule re-wrapped an already-a-DataFrame in pd.DataFrame(...)
   -- harmless but redundant (optimize_schedule already returns one); removed.

8. SELF-INFLICTED MERGE COLLISION, found by actually running this: an early version
   of walk_forward_forecast_eval merged forecast_df (which already carries
   demand_proxy as the ground truth alongside forecast_p50) against df's
   demand_proxy again, colliding column names into demand_proxy_x/demand_proxy_y
   and raising `KeyError: 'demand_proxy'` the moment the code tried to read it back.
   Fixed by using forecast_df's own demand_proxy directly -- the merge was
   unnecessary in the first place.
"""

import sys
import numpy as np
import pandas as pd

# clean_pipeline, forecast_workload, build_base_stats, optimize_schedule,
# DEFAULT_CONFIG are already in scope from the Phase 6 cell above


# ============================================================================
# SHARED COST LOGIC -- same dispatch the MILP itself uses, so the backtest
# evaluates schedules with the same formula that produced them
# ============================================================================
def _capacity_provided(department, workers, overtime_min, smv, shift_min):
    if department == "sewing":
        return (workers * shift_min + overtime_min) / smv
    return overtime_min  # finishing: demand_proxy is already in minutes, no SMV term


def _backlog_penalty(department, config):
    return config["backlog_penalty_per_piece"] if department == "sewing" else config["backlog_penalty_per_min"]


# ============================================================================
# STEP 1 — what actually happened (management's real historical decisions)
# ============================================================================
def actual_historical_cost(df, config):
    df = df.sort_values(["team_id", "date"]).copy()
    records = []
    for team_id, g in df.groupby("team_id"):
        dept = g["department"].iloc[0]
        prev_backlog = 0.0
        for row in g.itertuples():
            capacity = _capacity_provided(dept, row.no_of_workers, row.over_time, row.smv, config["shift_min"])
            backlog = max(0.0, row.demand_proxy + prev_backlog - capacity)
            cost = config["overtime_cost_per_min"] * row.over_time + _backlog_penalty(dept, config) * backlog
            records.append({"team_id": team_id, "department": dept, "date": row.date,
                             "backlog": backlog, "cost": cost})
            prev_backlog = backlog
    hist_df = pd.DataFrame(records)
    return hist_df, float(hist_df["cost"].sum())


# ============================================================================
# STEP 2 — Layer B: optimizer value under perfect hindsight
# ============================================================================
def hindsight_backtest(df, config):
    """Feeds the ACTUAL realized demand_proxy to the optimizer as if it were the
    forecast -- isolates optimization value from forecast-error cost."""
    base_stats = build_base_stats(df)
    hindsight_demand = df[["team_id", "date", "demand_proxy"]].rename(
        columns={"demand_proxy": config["demand_column"]}
    )
    result = optimize_schedule(hindsight_demand, base_stats, config)
    if result["status"] != "Optimal":
        raise RuntimeError(f"Hindsight backtest did not solve to optimality: {result['status']}")
    return result["total_cost"]


# ============================================================================
# STEP 3 — Layer A: forecast accuracy, walk-forward
# ============================================================================
def walk_forward_forecast_eval(df, n_folds=5, horizon=3):
    dates = sorted(df["date"].unique())
    fold_starts = np.linspace(int(len(dates) * 0.5), len(dates) - horizon - 1, n_folds).astype(int)

    results = []
    for start in fold_starts:
        cutoff = dates[start]
        forecast_df = forecast_workload(df, cutoff_date=cutoff)
        window_end_idx = min(start + horizon, len(dates) - 1)
        window_dates = dates[start + 1: window_end_idx + 1]
        window = forecast_df[forecast_df["date"].isin(window_dates)]
        if window.empty:
            continue

        # forecast_workload's output already carries demand_proxy (the ground
        # truth for that period) alongside forecast_p50 -- merging it again
        # from df would collide column names and silently produce
        # demand_proxy_x/demand_proxy_y instead of a usable column
        true, pred = window["demand_proxy"], window["forecast_p50"]
        mask = true > 0
        mape = float((true[mask] - pred[mask]).abs().div(true[mask]).mean()) if mask.any() else np.nan
        results.append({"cutoff": cutoff, "mape": mape, "n_obs": len(window),
                         "n_excluded_zero_true": int((~mask).sum())})

    return pd.DataFrame(results)


# ============================================================================
# STEP 4 — Layer C: end-to-end realistic backtest
# ============================================================================
def realized_cost_of_schedule(schedule_df, actual_df, config):
    merged = schedule_df.merge(
        actual_df[["team_id", "date", "demand_proxy", "smv"]], on=["team_id", "date"]
    ).sort_values(["team_id", "date"])
    if merged.empty:
        return 0.0

    records = []
    for team_id, g in merged.groupby("team_id"):
        dept = g["department"].iloc[0]
        prev_backlog = 0.0
        for row in g.itertuples():
            capacity = _capacity_provided(dept, row.workers, row.overtime_min, row.smv, config["shift_min"])
            backlog = max(0.0, row.demand_proxy + prev_backlog - capacity)
            cost = config["overtime_cost_per_min"] * row.overtime_min + _backlog_penalty(dept, config) * backlog
            records.append(cost)
            prev_backlog = backlog
    return float(sum(records))


def end_to_end_backtest(df, n_folds=5, horizon=3, config=None):
    config = config or DEFAULT_CONFIG
    dates = sorted(df["date"].unique())
    fold_starts = np.linspace(int(len(dates) * 0.5), len(dates) - horizon - 1, n_folds).astype(int)
    base_stats = build_base_stats(df)

    fold_results = []
    for start in fold_starts:
        cutoff = dates[start]
        forecast_df = forecast_workload(df, cutoff_date=cutoff)
        window_end_idx = min(start + horizon, len(dates) - 1)
        window_dates = dates[start + 1: window_end_idx + 1]
        window_forecast = forecast_df[forecast_df["date"].isin(window_dates)]
        if window_forecast.empty:
            continue

        result = optimize_schedule(window_forecast, base_stats, config)
        if result["status"] != "Optimal":
            # a non-optimal fold is a real finding, not something to silently
            # average in as if its cost number meant anything
            fold_results.append({"cutoff": cutoff, "status": result["status"],
                                  "realized_cost": np.nan, "actual_cost": np.nan})
            continue

        realized = realized_cost_of_schedule(result["schedule"], df, config)
        _, actual_cost = actual_historical_cost(df[df["date"].isin(window_dates)], config)
        fold_results.append({"cutoff": cutoff, "status": result["status"],
                              "realized_cost": realized, "actual_cost": actual_cost})

    return pd.DataFrame(fold_results)


# ============================================================================
# MAIN
# ============================================================================

output_dir = OUTPUT_DIR
import os
os.makedirs(output_dir, exist_ok=True)

# df reused directly from the Phase 1-3 cell above

# --- Step 1 ---
hist_df, hist_total_cost = actual_historical_cost(df, DEFAULT_CONFIG)
print(f"Actual historical cost (all teams): {hist_total_cost:,.0f}")

by_dept = hist_df.groupby("department").agg(cost=("cost", "sum"), backlog=("backlog", "sum"))
print("\nHistorical cost/backlog by department:")
print(by_dept.round(1).to_string())
if by_dept.loc["finishing", "backlog"] == 0:
    print("\nNote: finishing backlog is exactly 0 -- expected, not a finding. Finishing's "
          "demand_proxy IS over_time (no independent WIP signal exists for that department), "
          "so 'demand' and 'capacity provided' are definitionally the same number there. Any "
          "backlog-based savings story for finishing would be circular; treat sewing's figures "
          "as the meaningful ones for that part of the narrative.")

if hist_total_cost <= 0:
    print("\nWARNING: historical cost is ~0 -- the opportunity for savings is genuinely small. "
          "Report that rather than tuning cost-ratio assumptions until a bigger number appears.")

# --- Step 2: Layer B ---
hindsight_cost = hindsight_backtest(df, DEFAULT_CONFIG)
hindsight_savings_pct = (1 - hindsight_cost / hist_total_cost) * 100
print(f"\nOptimizer cost (perfect hindsight, full {df['date'].nunique()}-day period): {hindsight_cost:,.0f}")
print(f"Layer B — full-period hindsight ceiling: {hindsight_savings_pct:.1f}%")

# --- Step 3: Layer A ---
wf_results = walk_forward_forecast_eval(df)
print("\nWalk-forward forecast evaluation:")
print(wf_results.to_string(index=False))
print(f"Layer A — mean walk-forward MAPE: {wf_results['mape'].mean():.1%} "
      f"(std: {wf_results['mape'].std():.1%})")

# --- Step 4: Layer C ---
e2e = end_to_end_backtest(df)
e2e_valid = e2e[e2e["status"] == "Optimal"].copy()
if len(e2e_valid) < len(e2e):
    print(f"\nNote: {len(e2e) - len(e2e_valid)} of {len(e2e)} end-to-end folds did not solve "
          f"to optimality and are excluded from the savings estimate below -- an infeasible "
          f"fold under a realistic forecast is itself a finding about staffing-bound tightness.")
e2e_valid["savings_pct"] = (1 - e2e_valid["realized_cost"] / e2e_valid["actual_cost"]) * 100
print("\nEnd-to-end backtest by fold:")
print(e2e_valid.to_string(index=False))

mean_e2e = e2e_valid["savings_pct"].mean()
std_e2e = e2e_valid["savings_pct"].std()
print(f"\nLayer C — mean end-to-end savings: {mean_e2e:.1f}% (std: {std_e2e:.1f}%)")

if std_e2e is not None and not np.isnan(std_e2e) and std_e2e < 1e-6 and len(e2e_valid) > 1:
    print("WARNING: savings_pct is suspiciously identical across folds -- check that "
          "forecast_workload's cutoff_date is actually being respected and no future data "
          "is leaking into earlier folds.")

# --- Layer B, restricted to the SAME days as the Layer C backtest folds ---
# Comparing a whole-dataset hindsight ceiling against a partial-days realistic
# result isn't apples-to-apples (see fix #9) -- this is the number that
# Layer C should actually be checked against.
dates_all = sorted(df["date"].unique())
fold_starts = np.linspace(int(len(dates_all) * 0.5), len(dates_all) - 3 - 1, 5).astype(int)
backtest_days = sorted(set(
    d for start in fold_starts
    for d in dates_all[start + 1: min(start + 3, len(dates_all) - 1) + 1]
))
df_matched = df[df["date"].isin(backtest_days)]
hist_df_matched, hist_cost_matched = actual_historical_cost(df_matched, DEFAULT_CONFIG)
hindsight_cost_matched = hindsight_backtest(df_matched, DEFAULT_CONFIG)
hindsight_savings_matched = (1 - hindsight_cost_matched / hist_cost_matched) * 100
print(f"\nLayer B (scope-matched to the {len(backtest_days)} backtest days): "
      f"{hindsight_savings_matched:.1f}%")

if mean_e2e > hindsight_savings_matched + 1e-6:
    print(f"WARNING: Layer C ({mean_e2e:.1f}%) exceeds scope-matched Layer B "
          f"({hindsight_savings_matched:.1f}%) -- perfect information should never do worse "
          f"than a realistic forecast. Investigate before trusting either number.")
else:
    print(f"Sanity check OK: Layer C ({mean_e2e:.1f}%) <= scope-matched Layer B "
          f"({hindsight_savings_matched:.1f}%), as expected.")

# --- Step 5: summary ---
summary = pd.DataFrame([
    {"Layer": "B — Hindsight ceiling (full period)", "Result": f"{hindsight_savings_pct:.1f}% savings"},
    {"Layer": "B — Hindsight (scope-matched to backtest)", "Result": f"{hindsight_savings_matched:.1f}% savings"},
    {"Layer": "A — Forecast accuracy (walk-forward)", "Result": f"{wf_results['mape'].mean():.1%} mean MAPE"},
    {"Layer": "C — End-to-end (realistic)", "Result": f"{mean_e2e:.1f}% savings (std {std_e2e:.1f}%)"},
])
print("\n" + "=" * 60)
print(summary.to_string(index=False))
print("=" * 60)
print(f"\nHeadline sentence: on the same days used for backtesting, hindsight optimization "
      f"could save ~{hindsight_savings_matched:.0f}%, but realistic forecast error narrows "
      f"that to ~{mean_e2e:.0f}% achievable today (±{std_e2e:.0f}pp across a "
      f"{len(e2e_valid)}-fold walk-forward backtest on ~{df['date'].nunique()} days of data -- "
      f"directional, not statistically powered; would narrow with a full year of production "
      f"history).")

hist_df.to_csv(f"{output_dir}/phase9_historical_cost.csv", index=False)
wf_results.to_csv(f"{output_dir}/phase9_walk_forward_mape.csv", index=False)
e2e_valid.to_csv(f"{output_dir}/phase9_end_to_end_backtest.csv", index=False)
print(f"\nSaved detail tables to {output_dir}/")


Actual historical cost (all teams): 22,416,865

Historical cost/backlog by department:
                  cost    backlog
department                       
finishing    1455120.0        0.0
sewing      20961745.4  2843198.1

Note: finishing backlog is exactly 0 -- expected, not a finding. Finishing's demand_proxy IS over_time (no independent WIP signal exists for that department), so 'demand' and 'capacity provided' are definitionally the same number there. Any backlog-based savings story for finishing would be circular; treat sewing's figures as the meaningful ones for that part of the narrative.



Optimizer cost (perfect hindsight, full 59-day period): 12,535,635
Layer B — full-period hindsight ceiling: 44.1%



Walk-forward forecast evaluation:
    cutoff     mape  n_obs  n_excluded_zero_true
2015-02-04 0.336281     52                     0
2015-02-11 3.556960     49                     0
2015-02-19 0.089361     51                     0
2015-02-28 0.513966     56                     0
2015-03-08 0.191900     66                    10
Layer A — mean walk-forward MAPE: 93.8% (std: 147.3%)



End-to-end backtest by fold:
    cutoff  status  realized_cost   actual_cost  savings_pct
2015-02-04 Optimal   64524.782454 357816.199029    81.967059
2015-02-11 Optimal   40077.023292 307049.782709    86.947712
2015-02-19 Optimal   82306.279617 333106.164552    75.291277
2015-02-28 Optimal  236789.845968 434996.308084    45.565091
2015-03-08 Optimal  112808.620136 378458.110884    70.192574

Layer C — mean end-to-end savings: 72.0% (std: 16.1%)

Layer B (scope-matched to the 15 backtest days): 77.2%
Sanity check OK: Layer C (72.0%) <= scope-matched Layer B (77.2%), as expected.

                                    Layer                    Result
      B — Hindsight ceiling (full period)             44.1% savings
B — Hindsight (scope-matched to backtest)             77.2% savings
     A — Forecast accuracy (walk-forward)           93.8% mean MAPE
               C — End-to-end (realistic) 72.0% savings (std 16.1%)

Headline sentence: on the same days used for backtesting, hindsight opt